In [1]:
import gdown
import zipfile
import os
import shutil

In [2]:
# Google Drive link
file_url = "https://drive.google.com/file/d/1oUX991nPh4330ZzbXzBKKOAswhx75BtV/view?usp=sharing"

# Output filename
output = "dataset.zip"


In [3]:
# Download directly from Google Drive
gdown.download(file_url, output, quiet = False, fuzzy = True)

# Extract the zip file
extract_dir = "/content/images"
with zipfile.ZipFile(output, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

Downloading...
From (original): https://drive.google.com/uc?id=1oUX991nPh4330ZzbXzBKKOAswhx75BtV
From (redirected): https://drive.google.com/uc?id=1oUX991nPh4330ZzbXzBKKOAswhx75BtV&confirm=t&uuid=6c5f3fef-68eb-475e-a323-8f4e710fb4d1
To: /content/dataset.zip
100%|██████████| 9.26G/9.26G [01:47<00:00, 86.5MB/s]


Loading images

In [4]:
import random
import zipfile
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing import image
import re

In [5]:
image_dir = "/content/images/images"

In [33]:
# pattern = re.compile(r"^[a-z]+_\d+\.(jpg|jpeg|png)$", re.IGNORECASE)

# deleted_files = 0

# for fname in os.listdir(image_dir):
#     fpath = os.path.join(image_dir, fname)
#     if os.path.isfile(fpath) and pattern.match(fname):
#         os.remove(fpath)
#         deleted_files += 1

# print(f"🗑️ Deleted {deleted_files} mapped/renamed files.")

🗑️ Deleted 5097 mapped/renamed files.


In [ ]:
print("Total files extracted:", sum(len(files) for _, _, files in os.walk(image_dir)))
print("Sample files:", os.listdir(image_dir)[:10])

In [7]:
image_files = [f for f in os.listdir(image_dir) if f.endswith(".jpg") or f.endswith(".png")]

In [ ]:
plt.figure(figsize = (10, 10))
# Select a subset of image files to display
sample_files = random.sample(image_files, min(25, len(image_files)))
for i, file in enumerate(sample_files):
  img_path = os.path.join(image_dir, file)
  img = image.load_img(img_path, target_size = (200, 200))  # Adjust the target size as needed
  plt.subplot(5, 5, i + 1)
  plt.xticks([])
  plt.yticks([])
  plt.grid(False)
  plt.imshow(img)
plt.show()

In [8]:
from PIL import Image
from tqdm import tqdm
import hashlib
import pandas as pd

In [9]:
input_dir = "/content/images/images"

Checking for corrupted images

In [ ]:
bad_files = []

# Collect all image file paths
image_files = []
for dirpath, _, filenames in os.walk(image_dir):
    for f in filenames:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_files.append(os.path.join(dirpath, f))  # full path!

# Check each image
for path in tqdm(image_files, desc="Checking for corrupted images"):
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception:
        bad_files.append(path)
        try:
            os.remove(path)  # delete corrupted file
        except FileNotFoundError:
            pass  # file already gone

print(f"✅ Removed {len(bad_files)} corrupted images")

Checking for corrupted images: 100%|██████████| 34731/34731 [00:43<00:00, 805.20it/s] 

✅ Removed 0 corrupted images


Checking for duplicates

In [ ]:
seen = set()
duplicates = []

for path in tqdm(image_files, desc = "Checking for duplicates"):
    try:
        with open(path, "rb") as f:
            hash_val = hashlib.md5(f.read()).hexdigest()
        if hash_val in seen:
            os.remove(path)
            duplicates.append(path)
        else:
            seen.add(hash_val)
    except Exception as e:
        print(f"❌ Error processing {os.path.basename(path)}: {e}")

print(f"✅ Removed {len(duplicates)} duplicate images")

Checking for duplicates: 100%|██████████| 34731/34731 [01:34<00:00, 367.83it/s]

✅ Removed 0 duplicate images


Removing Outliers

In [ ]:
min_size = 50  # min width/height
max_size = 5000  # max width/height

for root, _, files in os.walk(input_dir):
    for file in files:
        path = os.path.join(root, file)
        try:
            img = Image.open(path)
            if img.width < min_size or img.height < min_size or img.width > max_size or img.height > max_size:
                os.remove(path)
        except:
            pass

In [40]:
# pick first 10 image paths
image_files = []
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith((".jpg")):
            image_files.append(os.path.join(root, file))

In [ ]:
# check sizes
for img_path in image_files:
    with Image.open(img_path) as img:
        print(f"{os.path.basename(img_path)} → {img.size}")  # (width, height)


In [ ]:
image_files = []
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith((".jpg")):
            image_files.append(os.path.join(root, file))

In [ ]:
# check sizes
for img_path in image_files:
    with Image.open(img_path) as img:
        print(f"{os.path.basename(img_path)} → {img.size}")  # (width, height)


Preparation Toward Model training

In [12]:
train = pd.read_csv("Train.csv")

In [11]:
train.head()

NameError: name 'train' is not defined

In [13]:
len(train)

26068

In [14]:
import shutil
from pathlib import Path

Mapping Damage type to images

In [15]:
filename_col = "filename"
damage_col = "damage"

In [16]:
# Go through each row and rename files
renamed_count = 0
for _, row in train.iterrows():
    old_name = str(row[filename_col]).lower()
    damage = str(row[damage_col]).replace(" ", "_").lower()  # safe folder name

    old_path = os.path.join(image_dir, old_name)

    if os.path.exists(old_path):
        # New filename: damageType_index.ext
        ext = os.path.splitext(old_name)[1]  # keep .jpg/.png
        new_name = f"{damage}_{renamed_count}{ext}"
        new_path = os.path.join(image_dir, new_name)

        os.rename(old_path, new_path)
        renamed_count += 1

print(f"✅ Done! Renamed {renamed_count} images in {image_dir}")

✅ Done! Renamed 26068 images in /content/images/images


In [ ]:
print("Sample files:", os.listdir(image_dir)[:10])

Moving Mapped images into a new folder

In [18]:
output_dir = "dataset_split/mapped"

In [19]:
os.makedirs(output_dir, exist_ok = True)

In [20]:
moved_count, missing_count = 0, 0

for idx, row in train.iterrows():
    damage = str(row[damage_col]).replace(" ", "_").lower()
    ext = os.path.splitext(str(row[filename_col]))[1].lower()
    new_name = f"{damage}_{idx}{ext}"
    src = os.path.join(image_dir, new_name)
    dst = os.path.join(output_dir, new_name)

    if os.path.exists(src):
        shutil.move(src, dst)
        moved_count += 1
    else:
        missing_count += 1

print(f"✅ Done! Moved {moved_count} mapped (renamed) files into {output_dir}")
print(f"⚠️ Missing {missing_count} files (listed in CSV but not found on disk)")

✅ Done! Moved 26068 mapped (renamed) files into dataset_split/mapped
⚠️ Missing 0 files (listed in CSV but not found on disk)


Confiming the total number of images mapped and moved to the new folder

In [21]:
len(train)

26068

In [22]:
mapped_dir = "/content/dataset_split/mapped"

print("Total mapped files:", sum(len(files) for _, _, files in os.walk(mapped_dir)))

Total mapped files: 26068


In [52]:
# if os.path.exists(mapped_dir):
#     shutil.rmtree(mapped_dir)
#     print(f"🗑️ Deleted folder: {mapped_dir}")
# else:
#     print(f"⚠️ Folder not found: {mapped_dir}")

🗑️ Deleted folder: /content/dataset_split/mapped


In [ ]:
print("Sample files:", os.listdir(mapped_dir)[:10])

In [ ]:
# def preview_images(start_index, n = 6):
#     rows = train.iloc[start_index:start_index+int(n)]

#     fig, axes = plt.subplots(3, 2, figsize = (15, 8))
#     axes = axes.flatten()

#     for ax, (_, row) in zip(axes, rows.iterrows()):
#         fname = str(row[filename_col]).lower()
#         img_path = os.path.join(image_dir, fname)

#         if os.path.exists(img_path):
#             img = mpimg.imread(img_path)
#             ax.imshow(img)
#             ax.axis("off")

#             # ✅ Use the correct damage column here:
#             ax.set_title(f"Damage: {row[damage_col]}", fontsize = 10, color = "black")
#         else:
#             ax.axis("off")
#             ax.set_title("⚠️ Not Found", fontsize = 10, color = "gray")

#     plt.tight_layout()
#     plt.show()

In [24]:
dataset_dir = "/content/dataset_split/mapped"
split_dir = "/content/dataset"

In [25]:
# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

In [26]:
mapped_dir = "/content/dataset_split/mapped"

print("Total mapped files:", sum(len(files) for _, _, files in os.walk(mapped_dir)))

Total mapped files: 26068


In [27]:
os.makedirs(split_dir, exist_ok = True)

In [28]:
import shutil
import random
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [29]:
# Spliting the images

data = []
for f in os.listdir(mapped_dir):
    if f.endswith((".jpg", ".jpeg", ".png")):
        label = f.split("_")[0]
        data.append((f, label))

df = pd.DataFrame(data, columns=["filename", "label"])


train_files, temp_files = train_test_split(df, train_size = train_ratio, stratify  = df["label"], random_state = 42)


relative_val_size = val_ratio / (1 - train_ratio)
val_files, test_files = train_test_split(temp_files, train_size = relative_val_size, stratify = temp_files["label"], random_state = 42)

splits = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

for split_name, split_df in splits.items():
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Moving {split_name} images"):
        label = row["label"]
        fname = row["filename"]

        src = os.path.join(mapped_dir, fname)
        dst_dir = os.path.join(split_dir, split_name, label)
        os.makedirs(dst_dir, exist_ok = True)
        dst = os.path.join(dst_dir, fname)

        if os.path.exists(src):
            shutil.move(src, dst)

print("✅ Dataset split completed!")
print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

Moving test images: 100%|██████████| 3911/3911 [00:00<00:00, 7835.88it/s]

✅ Dataset split completed!
Train: 18247 | Val: 3910 | Test: 3911


In [30]:
train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"
test_dir = "/content/dataset/test"
print("Total mapped files:", sum(len(files) for _, _, files in os.walk(train_dir)))
print("Total mapped files:", sum(len(files) for _, _, files in os.walk(val_dir)))
print("Total mapped files:", sum(len(files) for _, _, files in os.walk(test_dir)))

Total mapped files: 18247
Total mapped files: 3910
Total mapped files: 3911


Normalize Pixel Values

In [31]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

Data Augmentation on the train and val

In [32]:
from collections import Counter
import tensorflow as tf
import numpy as np

In [33]:
class_counts = {}
for cls in os.listdir(train_dir):
    cls_dir = os.path.join(train_dir, cls)
    if os.path.isdir(cls_dir):
        count = len([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        class_counts[cls] = count

print("Class distribution:")
for cls, count in class_counts.items():
    print(f"{cls}: {count}")


Class distribution:
g: 8136
wd: 6466
nd: 191
other: 293
dr: 3161


In [34]:
threshold = 6000
low_classes = [cls for cls, count in class_counts.items() if count < threshold]

print("Low classes (to augment):", low_classes)


Low classes (to augment): ['nd', 'other', 'dr']


In [35]:
# Defining my augmentation generator
augmenter = ImageDataGenerator(
    rotation_range = 15,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    zoom_range = 0.1,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = "nearest"
)

In [36]:
target_size = 300
augmented_per_class = 9000

for cls in low_classes:
    cls_dir = os.path.join(train_dir, cls)
    images = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    n_existing = len(images)
    n_to_generate = augmented_per_class - n_existing
    print(f"Augmenting {cls}: {n_existing} → {augmented_per_class} (generating {n_to_generate})")

    if n_to_generate > 0:
        # Load images and generate augmented ones
        for i, img_name in enumerate(images):
            img_path = os.path.join(cls_dir, img_name)
            img = tf.keras.utils.load_img(img_path, target_size = (target_size, target_size))
            x = tf.keras.utils.img_to_array(img)
            x = np.expand_dims(x, axis=0)

            gen = augmenter.flow(x, batch_size = 1, save_to_dir = cls_dir, save_prefix = f"aug_{cls}", save_format = "jpg")

            # Generate until class reaches target count
            for _ in range(n_to_generate // len(images) + 1):
                next(gen)
                n_to_generate -= 1
                if n_to_generate <= 0:
                    break

Augmenting nd: 191 → 9000 (generating 8809)
Augmenting other: 293 → 9000 (generating 8707)
Augmenting dr: 3161 → 9000 (generating 5839)


In [37]:
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.7, 1.3],
    fill_mode = "nearest"
)

In [38]:
target_size = (300, 300)

resized_count = 0

# Go through each class folder
for cls in os.listdir(train_dir):
    cls_dir = os.path.join(train_dir, cls)

    if os.path.isdir(cls_dir):
        # Loop over images in class
        for img_name in tqdm(os.listdir(cls_dir), desc = f"Resizing {cls}"):
            img_path = os.path.join(cls_dir, img_name)

            try:
                with Image.open(img_path).convert("RGB") as img:
                    img = img.resize(target_size, Image.Resampling.LANCZOS)
                    img.save(img_path, "JPEG")  # overwrite in place
                    resized_count += 1
            except Exception as e:
                print(f"❌ Error resizing {img_name}: {e}")

print(f"✅ Done! Resized {resized_count} images in training set to {target_size}")


Resizing dr: 100%|██████████| 6843/6843 [02:17<00:00, 49.84it/s]

✅ Done! Resized 30520 images in training set to (300, 300)


In [39]:
from tensorflow.keras import layers, models, regularizers

Building CNN model using 5 classes

In [49]:

num_classes = 5
weight_decay = 1e-1

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation = "relu", input_shape = (300, 300, 3),
                  kernel_regularizer = regularizers.l2(weight_decay)),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.3),

    layers.Conv2D(64, (3,3), activation = "relu",
                  kernel_regularizer = regularizers.l2(weight_decay)),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.3),

    layers.Conv2D(128, (3,3), activation = "relu",
                  kernel_regularizer = regularizers.l2(weight_decay)),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.4),

    layers.Flatten(),
    layers.Dense(128, activation = "relu",
                 kernel_regularizer = regularizers.l2(weight_decay)),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation = "softmax")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compiling Model Using Adam

In [50]:
model.compile(optimizer = "adam",
              loss = "categorical_crossentropy",
              metrics = ["accuracy"])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 298, 298, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 149, 149, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 149, 149, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 147, 147, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 73, 73, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 73, 73, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 71, 71, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 35, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 35, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 156800)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    20,070,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,164,421 (76.92 MB)

 Trainable params: 20,164,421 (76.92 MB)

 Non-trainable params: 0 (0.00 B)

Model Fitting

In [51]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size = (300, 300),
    batch_size = 32,
    class_mode = "categorical"
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size = (300, 300),
    batch_size = 32,
    class_mode = "categorical"
)

Found 30520 images belonging to 5 classes.
Found 3910 images belonging to 5 classes.


In [52]:
classes = sorted(os.listdir(train_dir))

print(f"Found {len(classes)} classes:")
for cls in classes:
    class_dir = os.path.join(train_dir, cls)
    if os.path.isdir(class_dir):
        count = len([f for f in os.listdir(class_dir) if f.endswith((".jpg", ".jpeg", ".png"))])
        print(f" - {cls}: {count} images")

Found 5 classes:
 - dr: 6843 images
 - g: 8136 images
 - nd: 4474 images
 - other: 4601 images
 - wd: 6466 images


In [53]:
import os

val_dir = "dataset/val"
for cls in os.listdir(val_dir):
    cls_path = os.path.join(val_dir, cls)
    if os.path.isdir(cls_path):
        print(f"{cls}: {len(os.listdir(cls_path))} images")

g: 1743 images
wd: 1386 images
nd: 41 images
other: 63 images
dr: 677 images


In [54]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
early_stop = EarlyStopping(monitor = "val_loss", patience = 5, restore_best_weights = True)
lr_reduce = ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 3)


history = model.fit(train_generator,
                    epochs = 20,
                    validation_data = val_generator,
                    callbacks = [early_stop, lr_reduce])

Epoch 1/20
  3/954 ━━━━━━━━━━━━━━━━━━━━ 2:05:26 8s/step - accuracy: 0.2188 - loss: 14.3461

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(np.argmax(labels, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))
